In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_huggingface import HuggingFaceEmbeddings
from langchain.prompts import PromptTemplate

from langchain.chains import RetrievalQA



In [7]:
## Read the ppdfs from the folder
loader=PyPDFDirectoryLoader("./us_census")

documents=loader.load()

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)

final_documents=text_splitter.split_documents(documents)
final_documents[0]

Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.2 (Windows)', 'creationdate': '2023-09-09T07:52:17-04:00', 'author': 'U.S. Census Bureau', 'keywords': 'acsbr-015', 'moddate': '2023-09-12T14:44:47+01:00', 'title': 'Health Insurance Coverage Status and Type by Geography: 2021 and 2022', 'trapped': '/false', 'source': 'us_census\\acsbr-015.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1'}, page_content='Health Insurance Coverage Status and Type \nby Geography: 2021 and 2022\nAmerican Community Survey Briefs\nACSBR-015\nIssued September 2023\nDouglas Conway and Breauna Branch\nINTRODUCTION\nDemographic shifts as well as economic and govern-\nment policy changes can affect people’s access to \nhealth coverage. For example, between 2021 and 2022, \nthe labor market continued to improve, which may \nhave affected private coverage in the United States \nduring that time.1 Public policy changes included \nthe renewal of the Public Health Emergency, wh

In [8]:
len(final_documents)

316

In [ ]:
## Embedding Using Huggingface
huggingface_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",       #sentence-transformers/all-MiniLM-l6-v2
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

c:\Users\saiar\Documents\0.GENAI\GenAI\venv\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\saiar\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [11]:
import  numpy as np
print(np.array(huggingface_embeddings.embed_query(final_documents[0].page_content)))
print(np.array(huggingface_embeddings.embed_query(final_documents[0].page_content)).shape)

[-8.30601528e-02 -1.45066706e-02 -2.10276805e-02  2.72682551e-02
  4.53647189e-02  5.28341383e-02 -2.53759213e-02  3.61304022e-02
 -9.08312425e-02 -2.77017597e-02  7.97397718e-02  6.42474815e-02
 -3.54004018e-02 -4.04245928e-02 -1.13772415e-02  4.45296019e-02
 -3.88542423e-03 -3.79060814e-03 -4.54510413e-02  2.67046951e-02
 -2.05681641e-02  2.87432466e-02 -2.41201185e-02 -3.69412303e-02
  1.92781053e-02  1.06194559e-02  3.21826804e-03  2.33249855e-03
 -4.29321602e-02 -1.64999202e-01  2.77008908e-03  2.68276669e-02
 -4.12894525e-02 -1.88446585e-02  1.58918444e-02  9.22320783e-03
 -2.00687721e-02  8.16561431e-02  3.89413126e-02  5.52223213e-02
 -3.69984470e-02  1.75319053e-02 -1.28966765e-02  2.80639302e-04
 -2.51580607e-02  4.59338212e-03 -2.39579398e-02 -5.76566067e-03
  6.02950901e-03 -3.61178257e-02  3.84415649e-02 -1.75470242e-03
  5.05656376e-02  6.02408908e-02  4.52067964e-02 -4.91435081e-02
  1.82053987e-02 -1.46668749e-02 -2.53130775e-02  3.18243764e-02
  5.15598208e-02 -9.32342

In [12]:
## VectorStore Creation
vectorstore=FAISS.from_documents(final_documents[:120],huggingface_embeddings)

In [13]:
## Query using Similarity Search
query="WHAT IS HEALTH INSURANCE COVERAGE?"
relevant_docments=vectorstore.similarity_search(query)

print(relevant_docments[0].page_content)

2 U.S. Census Bureau
WHAT IS HEALTH INSURANCE COVERAGE?
This brief presents state-level estimates of health insurance coverage 
using data from the American Community Survey (ACS). The  
U.S. Census Bureau conducts the ACS throughout the year; the 
survey asks respondents to report their coverage at the time of 
interview. The resulting measure of health insurance coverage, 
therefore, reflects an annual average of current comprehensive 
health insurance coverage status.* This uninsured rate measures a 
different concept than the measure based on the Current Population 
Survey Annual Social and Economic Supplement (CPS ASEC). 
For reporting purposes, the ACS broadly classifies health insurance 
coverage as private insurance or public insurance. The ACS defines 
private health insurance as a plan provided through an employer 
or a union, coverage purchased directly by an individual from an 
insurance company or through an exchange (such as healthcare.


In [14]:
retriever=vectorstore.as_retriever(search_type="similarity",search_kwargs={"k":3})
print(retriever)

tags=['FAISS', 'HuggingFaceEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000020165849350> search_kwargs={'k': 3}


In [15]:
from dotenv import load_dotenv
import os

load_dotenv()  # Load environment variables from .env

# Access the token
HUGGINGFACEHUB_API_TOKEN = os.environ["HUGGINGFACEHUB_API_TOKEN"]

The Hugging Face Hub is an platform with over 350k models, 75k datasets, and 150k demo apps (Spaces), all open source and publicly available, in an online platform where people can easily collaborate and build ML together.

In [34]:
from langchain_community.llms import HuggingFaceHub

hf=HuggingFaceHub(
    repo_id="mistralai/Mistral-7B-v0.1",
    model_kwargs={"temperature":0.1,"max_length":500}

)
query="What is the health insurance coverage?"
hf.invoke(query)

c:\Users\saiar\Documents\0.GENAI\GenAI\venv\Lib\site-packages\huggingface_hub\utils\_deprecation.py:131: FutureWarning: 'post' (from 'huggingface_hub.inference._client') is deprecated and will be removed from version '0.31.0'. Making direct POST requests to the inference server is not supported anymore. Please use task methods instead (e.g. `InferenceClient.chat_completion`). If your use case is not supported, please open an issue in https://github.com/huggingface/huggingface_hub.
  warnings.warn(warning_message, FutureWarning)


HfHubHTTPError: (Request ID: Root=1-68140a55-738c277d4e0dd00017abc671;38ecd61a-3323-4750-a6d9-556a1dcbca58)

403 Forbidden: None.
Cannot access content at: https://router.huggingface.co/hf-inference/models/mistralai/Mistral-7B-v0.1.
Make sure your token has the correct permissions.
The model mistralai/Mistral-7B-v0.1 is too large to be loaded automatically (14GB > 10GB).

In [35]:
#Hugging Face models can be run locally through the HuggingFacePipeline class.
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline

hf = HuggingFacePipeline.from_model_id(
    model_id="mistralai/Mistral-7B-v0.1",
    task="text-generation",
    pipeline_kwargs={"temperature": 0, "max_new_tokens": 300}
)

llm = hf 
llm.invoke(query)

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/mistralai/Mistral-7B-v0.1.
401 Client Error. (Request ID: Root=1-68140a60-08a4f40e447091dd552d4067;e7e764f7-ae57-452a-9408-5f34dc668dae)

Cannot access gated repo for url https://huggingface.co/mistralai/Mistral-7B-v0.1/resolve/main/config.json.
Access to model mistralai/Mistral-7B-v0.1 is restricted. You must have access to it and be authenticated to access it. Please log in.

In [36]:
prompt_template="""
Use the following piece of context to answer the question asked.
Please try to provide the answer only based on the context

{context}
Question:{question}

Helpful Answers:
 """

In [37]:
prompt=PromptTemplate(template=prompt_template,input_variables=["context","question"])

In [38]:
retrievalQA=RetrievalQA.from_chain_type(
    llm=hf,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt":prompt}
)

In [39]:
query="""DIFFERENCES IN THE
UNINSURED RATE BY STATE
IN 2022"""

In [40]:
# Call the QA chain with our query.
result = retrievalQA.invoke({"query": query})
print(result['result'])

c:\Users\saiar\Documents\0.GENAI\GenAI\venv\Lib\site-packages\huggingface_hub\utils\_deprecation.py:131: FutureWarning: 'post' (from 'huggingface_hub.inference._client') is deprecated and will be removed from version '0.31.0'. Making direct POST requests to the inference server is not supported anymore. Please use task methods instead (e.g. `InferenceClient.chat_completion`). If your use case is not supported, please open an issue in https://github.com/huggingface/huggingface_hub.
  warnings.warn(warning_message, FutureWarning)


HfHubHTTPError: (Request ID: Root=1-68140ab3-0dba12314da2e8c3571c4a06;fb98b104-85f2-4269-a1db-2c0347eff231)

403 Forbidden: None.
Cannot access content at: https://router.huggingface.co/hf-inference/models/mistralai/Mistral-7B-v0.1.
Make sure your token has the correct permissions.
The model mistralai/Mistral-7B-v0.1 is too large to be loaded automatically (14GB > 10GB).